# Train and inspect a model, step by step

This notebook does what `main.py` and `infer.py` do, one step per cell, so you can look at the data and change
the settings in between.

1. [Get the data and inspect it](#1.-Get-the-data-and-inspect-it)
2. [Train / validation split](#2.-Train-/-validation-split)
3. [Create the model](#3.-Create-the-model)
4. [Train](#4.-Train)
5. [Save the model](#5.-Save-the-model)
6. [Reload the model](#6.-Reload-the-model)
7. [Inference on new files](#7.-Inference-on-new-files)

**Before you start**

- Build the dataset in the app (*Data → Training dataset*) with `"normalize": "zscore"` and both NaN boxes
  ticked. See `README.md` in this folder.
- Run the notebook from this folder (`to_ignore/label_training`), so the scripts next to it can be imported.
- Sections 1 to 6 need `numpy`, `torch` and `matplotlib`. Section 7 reads raw files, which needs the
  SciFiReaders fork the app uses: pick the kernel of the repository's `.venv` for that.

In [ ]:
%matplotlib inline
import json
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

import data_io
import plots
import training_fns as tf
from main import build_config
from model import build_model, count_parameters, default_model_config

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Settings

Everything you may want to change is in this cell.

In [ ]:
# --- data
DATA_DIR = Path(r"PATH\TO\labeled\training_data")   # the folder with the built files
SET_NAME = None            # e.g. "binary_image"; None works when the folder holds one dataset
USE_AUX = True             # feed the scan size / bias range / mean / std values to the model
IMAGE_ORIGIN = "lower"     # how images are drawn: "lower" for Nanonis scans, "upper" for ordinary images

# --- split
VAL_FRACTION = 0.2
SEED = 42

# --- training
EPOCHS = 100
PATIENCE = 10              # stop after this many epochs without improvement of the validation loss
MIN_DELTA = 1e-4           # how much the validation loss must drop to count as an improvement
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
CLASS_WEIGHTS = True       # weight rare classes up (classification only)
DEVICE = "auto"            # "auto", "cpu", "cuda", "cuda:0" ...

# --- output
OUT_DIR = None             # None: DATA_DIR / "model"

# --- section 7: raw files to run the trained model on (files or folders)
INFER_FILES = []           # e.g. [r"D:\data\new_scans\scan_101.sxm", r"D:\data\new_scans"]

## 1. Get the data and inspect it

`load_dataset` finds the dataset, checks that all files are there and that it was built the way the model
expects, and loads the arrays. It stops with a message if something is missing.

In [ ]:
data = data_io.load_dataset(DATA_DIR, SET_NAME, use_aux=USE_AUX)
y, task = data_io.make_targets(data)
files = data_io.sample_files(DATA_DIR, data.prefix)        # file name of every sample, if meta.csv is there
source = data_io.source_summary(DATA_DIR, data.prefix)     # channel / flatten the samples were labeled with
classification = task["type"] == "classification"

print(f"dataset   {data.prefix}   (mode: {data.mode}, kind: {data.kind}, built: {data.info.get('built')})")
print(f"samples   {data.n}")
print(f"input     {data.x.shape}  dtype {data.x.dtype}   channels: {data_io.channel_names(data)}")
print(f"output    {data.y.shape}  ->  " + (
    "classes " + ", ".join(f"{c['label']} (class_id {c['class_id']}) = index {c['index']}" for c in task["classes"])
    if classification else f"score, mean {task['score_mean']:.4g}, std {task['score_std']:.4g}"))
print(f"aux       {None if data.aux is None else data.aux.shape}   columns: {data.aux_columns}")
print(f"resized   {data.info.get('transform')}")
if source:
    print(f"labeled   channel {source['channel']!r}, direction {source['direction']!r}, flatten {source['flatten']!r}"
          + ("" if source["consistent"] else "   <- NOT the same for all samples, see source['combinations']"))

The targets: how many samples per class, or how the scores are spread.

In [ ]:
plots.plot_targets(y, task);

A few samples, exactly as the model will see them (normalized, NaN set to 0). For images the NaN mask is drawn
below the data: white is data, black was NaN. Change `show` to look at other samples.

In [ ]:
show = list(range(min(6, data.n)))          # indices of the samples to draw
names = plots.target_names(y, task)
titles = [f"{i}: {names[i]}" + (f"\n{files[i]}" if files else "") for i in show]
plots.show_samples(data.kind, data.x[show], titles, origin=IMAGE_ORIGIN, bias_label="bias (V)");

Value ranges of the input channels. After the per-sample z-score the data channel should be around mean 0 and
std 1; a mask channel holds only 0 and 1.

In [ ]:
for c, name in enumerate(data_io.channel_names(data)):
    v = data.x[:, c]
    print(f"{name:14s} min {v.min():8.3f}   max {v.max():8.3f}   mean {v.mean():7.3f}   std {v.std():6.3f}")
mask_channels = [c for c, name in enumerate(data_io.channel_names(data)) if name.startswith("mask")]
print(f"\nshare of points that were NaN: {1 - data.x[:, mask_channels].mean():.1%}")

The aux values (z-scores over the dataset): one dot per sample. A column that is 0 everywhere had the same value
for every sample. If a column separates the classes by itself, the model can use that as a shortcut.

In [ ]:
if data.aux is not None:
    plots.plot_aux(data.aux, data.aux_columns, y, task)
    for name in data.aux_columns:
        s = data.norm["aux"]["columns"][name]
        print(f"{name:12s} log {str(s['log']):5s}  mean {s['mean']:.4g}  std {s['std']:.4g}" + ("   (constant)" if s["constant"] else ""))
else:
    print("aux values are not used (USE_AUX = False)")

## 2. Train / validation split

For classification the split is stratified: every class keeps about `VAL_FRACTION` of its samples for
validation. With the same `SEED` the split is always the same.

In [ ]:
tf.set_seed(SEED)
train_idx, val_idx = data_io.split_indices(data.n, y if classification else None, VAL_FRACTION, SEED)

x, preprocessing = data.x, {}
if data.kind == "spectrum":      # the bias row is in volts: scale it with the training samples' statistics
    x, preprocessing["bias_channel"] = data_io.scale_bias_channel(x, train_idx)
    print("bias row scaled with", preprocessing["bias_channel"])

print(f"training {len(train_idx)} samples, validation {len(val_idx)} samples")
if files:
    print("validation files:", [files[i] for i in val_idx][:12], "..." if len(val_idx) > 12 else "")
plots.plot_targets(y, task, train_idx, val_idx);

## 3. Create the model

`default_model_config` picks the settings for this kind of data: 2D convolutions for images, 1D for spectra, as
many input channels as a sample has, one output per class (or one number for a score). The dict is everything
that defines the network; change entries before building it if you like.

In [ ]:
device = tf.get_device(DEVICE)
n_aux = data.aux.shape[1] if data.aux is not None else 0
model_config = default_model_config(data.kind, x.shape[1:], n_aux, task["n_outputs"])

# examples of changes:
# model_config["channels"] = [32, 64, 128]
# model_config["dropout"] = 0.5

model = build_model(model_config)
print(json.dumps(model_config, indent=1))
print(f"\n{count_parameters(model)} parameters, device: {device}\n")
print(model)

## 4. Train

After every epoch the validation loss is checked. A `*` marks an epoch that improved it. When it has not improved
for `PATIENCE` epochs in a row the training stops, and the model goes back to the weights of the best epoch.

In [ ]:
weights = tf.class_weights(y[train_idx], task["n_outputs"]) if classification and CLASS_WEIGHTS else None
train_loader, val_loader = tf.make_loaders(x, data.aux, y, train_idx, val_idx, BATCH_SIZE, device, SEED)

started = time.time()
result = tf.train(model, train_loader, val_loader, task["type"], device, epochs=EPOCHS, patience=PATIENCE,
                  lr=LR, weight_decay=WEIGHT_DECAY, weights=weights, min_delta=MIN_DELTA)
seconds = time.time() - started
print(f"\n{result['epochs_run']} epochs in {seconds:.0f} s; best epoch {result['best_epoch']}, validation loss {result['best_val_loss']:.4f}")

In [ ]:
plots.plot_history(result["history"], result["best_epoch"]);

How good is the best model? For classification: the accuracy and the confusion matrix (rows are the true class,
columns what the model said). For scores: the error in the units of your scores.

In [ ]:
metrics = {"train": tf.evaluate(model, train_loader, task["type"], device, task),
           "validation": tf.evaluate(model, val_loader, task["type"], device, task)}
labels = [c["label"] for c in task.get("classes", [])]
for part, m in metrics.items():
    if classification:
        print(f"{part:10s} loss {m['loss']:.4f}   accuracy {m['accuracy']:.3f}")
    else:
        print(f"{part:10s} loss {m['loss']:.4f}   mean error {m['mae_score']:.3f}   rms error {m['rmse_score']:.3f}  (score units)")
if classification:
    plots.plot_confusion(metrics["validation"]["confusion_matrix"], labels)
else:
    plots.plot_scores(metrics["validation"]["true_scores"], metrics["validation"]["predicted_scores"])

The validation samples, one by one. For classification the wrong ones are listed first.

In [ ]:
aux_val = data.aux[val_idx] if data.aux is not None else None
out_val = tf.predict_arrays(model, x[val_idx], aux_val, task["type"], device)

if classification:
    pred = out_val.argmax(1)
    wrong = np.flatnonzero(pred != y[val_idx])
    order = list(wrong) + [k for k in range(len(val_idx)) if k not in set(wrong)]
    print(f"{len(wrong)} of {len(val_idx)} validation samples are wrong\n")
    for k in order[:20]:
        i = val_idx[k]
        print(f"{'WRONG' if k in wrong else 'ok   '}  sample {i:4d}  true {labels[y[i]]:10s} predicted {labels[pred[k]]:10s} "
              f"p = {out_val[k].max():.2f}   {files[i] if files else ''}")
    shown = order[:6]
    plots.show_samples(data.kind, x[val_idx[shown]],
                       [f"{val_idx[k]}: {labels[y[val_idx[k]]]} -> {labels[pred[k]]}" for k in shown], origin=IMAGE_ORIGIN);
else:
    true = y[val_idx] * task["score_std"] + task["score_mean"]
    predicted = out_val * task["score_std"] + task["score_mean"]
    order = np.argsort(-np.abs(predicted - true))
    for k in order[:20]:
        i = val_idx[k]
        print(f"sample {i:4d}  true {true[k]:8.3f}  predicted {predicted[k]:8.3f}  error {predicted[k] - true[k]:+7.3f}   {files[i] if files else ''}")

## 5. Save the model

Three files are written: `model.pt` (the weights with the configuration), `model_config.txt` (the configuration,
readable: data, network, training settings, the split, the results, and what is needed to treat new files the
same way) and `history.csv`.

In [ ]:
settings = {"epochs": EPOCHS, "patience": PATIENCE, "min_delta": MIN_DELTA, "batch_size": BATCH_SIZE, "lr": LR,
            "weight_decay": WEIGHT_DECAY, "seed": SEED, "val_fraction": VAL_FRACTION}
config = build_config(data, x, task, model, model_config, preprocessing, source, train_idx, val_idx, result,
                      metrics, weights, settings, device, DATA_DIR, seconds)
out_dir = tf.save_model(OUT_DIR or Path(DATA_DIR) / "model", model, config, result["history"])

print("saved in", out_dir, ":", sorted(p.name for p in out_dir.iterdir()))
print()
print("\n".join(config["description"]))

## 6. Reload the model

`load_model` rebuilds the network from the saved configuration and loads the weights. The check below compares
its outputs on the validation samples with those of the model still in memory: the difference should be 0.

In [ ]:
loaded, loaded_config = tf.load_model(out_dir / "model.pt", device)
out_loaded = tf.predict_arrays(loaded, x[val_idx], aux_val, loaded_config["task"]["type"], device)

print("sections of the saved configuration:", list(loaded_config))
print("largest difference between the reloaded and the trained model:", float(np.abs(out_loaded - out_val).max()))

## 7. Inference on new files

Each raw file is read and prepared exactly like a training sample: the same channel and flatten, resized the same
way, normalized with its own mean and std, and its aux values normalized with the statistics of the training
dataset (they are stored in the configuration). Set `INFER_FILES` in the settings cell.

This section needs the SciFiReaders fork the app uses (the repository's `.venv` kernel).

In [ ]:
import argparse
import infer

if not INFER_FILES:
    print("Set INFER_FILES in the settings cell to raw files or folders, then run this cell again.")
else:
    reader, dataset = infer.import_app()                     # the app's own reader and transforms
    recipe = infer.load_recipe(loaded_config)
    # to use another channel / flatten than the samples were labeled with, set them here instead of None
    chosen = infer.choose_source(recipe, argparse.Namespace(channel=None, direction=None, flatten=None, slice_index=None))
    print("files are read as:", chosen, "\n")

    loaded.eval()
    prepared, shown_titles = [], []
    for path in infer.collect_files(INFER_FILES, reader):
        try:
            xi, auxi, details = infer.prepare(path, recipe, chosen, reader, dataset)
            p = infer.predict(loaded, loaded_config, xi, auxi, device)
            text = f"score {p['score']:.3f}" if "score" in p else f"{p['prediction']} (class_id {p['class_id']}, p = {p['probability']:.2f})"
            notes = "; ".join(details["notes"])
            prepared.append(xi)
            shown_titles.append(f"{path.name}\n{text}")
        except ValueError as e:
            text, notes = "skipped", str(e)
        print(f"{path.name}: {text}" + (f"   [{notes}]" if notes else ""))

    if prepared:
        plots.show_samples(recipe["kind"], np.stack(prepared[:6]), shown_titles[:6], origin=IMAGE_ORIGIN)